In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_ViT
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [4]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_ViT()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("dense_16").output,
    outputs=model.output
)

In [8]:
l_label = [2,3,4,5,6,7,8,9,10,13]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_2/ViT-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    #x_attack = np.load(o_path, allow_pickle=True)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_2/ViT-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_2/ViT-8/layer_cache/salt/"+str(i))

[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/base
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/0
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/0
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/1
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/1
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/2
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/2
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/3
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/3
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/4
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/4
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/gauss/5
[SKIP] All layer files already exist in D:/Data_2/ViT-8/layer_cache/salt/5
[SKIP] All layer file

In [11]:
CACHE_DIR = "./ViT-8/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_2/ViT-8/layer_cache/base")


==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 18032
xi>=0 count: 16333
xi>=0 count: 14584
xi>=0 count: 13925
xi>=0 count: 13912
xi>=0 count: 13155
xi>=0 count: 12338
xi>=0 count: 13549
xi>=0 count: 16218

==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 17860
xi>=0 count: 16109
xi>=0 count: 14232
xi>=0 count: 14035
xi>=0 count: 15298
xi>=0 count: 14031
xi>=0 count: 13774
xi>=0 count: 14935
xi>=0 count: 16787

==== split 0, thr

In [13]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_2/ViT-8/layer_cache/base")

In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_2/ViT-8/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.88193417 0.7468736  0.63080814 0.60373168 0.62229233 0.50401399
 0.47676442 0.60623252 0.80005534]
Layer 1
accuracy: [0.89422617 0.78976055 0.68248635 0.61034859 0.66664615 0.58358716
 0.60836527 0.69283792 0.79791257]
Layer 2
accuracy: [0.91669244 0.80063455 0.681249   0.61461623 0.67114302 0.57612305
 0.59723622 0.67699918 0.79264744]
Layer 3
accuracy: [0.90566943 0.7931755  0.67945647 0.61841997 0.69654761 0.60763691
 0.58826511 0.64926135 0.78822147]
Layer 4
accuracy: [0.95067415 0.89423267 0.78566471 0.71599588 0.73449384 0.62847084
 0.60268477 0.68880135 0.77946551]
Layer 5
accuracy: [0.94607315 0.90618244 0.80001104 0.74421373 0.7436717  0.65954264
 0.66461312 0.72543483 0.82581374]
Layer 6
accuracy: [0.94960373 0.93543747 0.86471286 0.80369012 0.77285593 0.64892579
 0.67114819 0.74686879 0.83475357]
Layer 7
accuracy: [0.94629256 0.92413492 0.8152471  0.80231675 0.79317638 0.67630627
 0.68712692 0.78197879 0.85918462]
Layer 8
accuracy: [0.94728545 0.93502627

In [16]:
compute_stats(base)

(array([[0.7532053 , 0.57667933, 0.6276841 ],
        [0.78882436, 0.62019397, 0.69970525],
        [0.79952533, 0.62062743, 0.68896094],
        [0.79276714, 0.64086816, 0.67524931],
        [0.87685718, 0.69298685, 0.69031721],
        [0.88408888, 0.71580936, 0.73862056],
        [0.91658469, 0.74182395, 0.75092352],
        [0.89522486, 0.75726647, 0.77609678],
        [0.90816665, 0.80234632, 0.78961279],
        [0.93994651, 0.83385606, 0.85006545],
        [1.        , 0.99984582, 0.99992787]]),
 array([[1.02619489e-01, 5.19378725e-02, 1.32851751e-01],
        [8.64449539e-02, 3.46159657e-02, 7.75345711e-02],
        [9.61225794e-02, 3.90239194e-02, 8.02234332e-02],
        [9.23515052e-02, 3.96166568e-02, 8.36746334e-02],
        [6.84760781e-02, 4.62405355e-02, 7.21783928e-02],
        [6.16421361e-02, 3.97871930e-02, 6.64670730e-02],
        [3.71320650e-02, 6.68841669e-02, 6.68531294e-02],
        [5.72717014e-02, 5.73689894e-02, 7.03652928e-02],
        [4.69214766e-02, 2.7

In [17]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/ViT-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.93356908 0.81918291 0.73829157 0.64354996 0.61307123 0.61516327
 0.72162119 0.83069739 0.91957748]
Layer 1
accuracy: [0.9202713  0.81380963 0.71439262 0.61549823 0.63962053 0.6150059
 0.66252386 0.76213653 0.86553528]
Layer 2
accuracy: [0.93610568 0.81895105 0.69946068 0.61499769 0.64480507 0.59861397
 0.6354315  0.73284134 0.86890563]
Layer 3
accuracy: [0.93613769 0.81248667 0.70298575 0.61809833 0.66130058 0.60304519
 0.60253141 0.68970574 0.86163733]
Layer 4
accuracy: [0.94649889 0.85947371 0.75059087 0.67552724 0.66449761 0.60901241
 0.59863347 0.70186952 0.82444039]
Layer 5
accuracy: [0.9425552  0.86482321 0.75336715 0.68164564 0.66601885 0.61671239
 0.64239808 0.71006913 0.84686057]
Layer 6
accuracy: [0.93816171 0.87453006 0.78511422 0.70691758 0.67723237 0.58714542
 0.63243093 0.71480379 0.84938029]
Layer 7
accuracy: [0.94086149 0.86917471 0.7538943  0.6988316  0.68276526 0.60580667
 0.65455272 0.76041257 0.87375226]
Layer 8
accuracy: [0.93837285 0.86198276 

In [18]:
compute_stats(gauss)

(array([[0.83041405, 0.62495038, 0.82401657],
        [0.81788829, 0.6244015 , 0.76483591],
        [0.82122469, 0.61948941, 0.7477542 ],
        [0.81745694, 0.62833375, 0.72101467],
        [0.8514417 , 0.64961195, 0.70998096],
        [0.85255678, 0.65630436, 0.73549259],
        [0.86331077, 0.65990879, 0.73569377],
        [0.85332056, 0.66623759, 0.76481816],
        [0.85282497, 0.69211519, 0.77494906],
        [0.85732573, 0.68544734, 0.79034465],
        [0.91249592, 0.75854267, 0.8758923 ]]),
 array([[0.08045228, 0.01360408, 0.08089539],
        [0.08345656, 0.01121892, 0.08390614],
        [0.09431938, 0.0188909 , 0.09557798],
        [0.09438073, 0.02352309, 0.10663977],
        [0.08049547, 0.02847976, 0.09248176],
        [0.07727324, 0.0278318 , 0.08571988],
        [0.06477389, 0.04948488, 0.09107851],
        [0.07729697, 0.04199614, 0.09374703],
        [0.07112003, 0.02270261, 0.10009245],
        [0.07125229, 0.0135058 , 0.08626174],
        [0.05590616, 0.01399328,

In [19]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/ViT-8/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.92943486 0.81445569 0.71392847 0.6321724  0.62248283 0.60483024
 0.70276691 0.82381262 0.91764569]
Layer 1
accuracy: [0.91129877 0.80808041 0.71694771 0.61812592 0.65858697 0.61529841
 0.64700225 0.73436092 0.82957072]
Layer 2
accuracy: [0.93066045 0.81829534 0.70639056 0.61640982 0.65595899 0.59920093
 0.62681534 0.70061416 0.81311701]
Layer 3
accuracy: [0.92255484 0.8149631  0.70965934 0.62502431 0.67154333 0.6163982
 0.60892304 0.66891857 0.8053019 ]
Layer 4
accuracy: [0.95044079 0.88540221 0.77627477 0.69944711 0.69012049 0.62216188
 0.60614545 0.68191615 0.78414074]
Layer 5
accuracy: [0.94707616 0.8866329  0.78108689 0.70752422 0.70119538 0.63420863
 0.65190763 0.70454974 0.82082553]
Layer 6
accuracy: [0.94212464 0.90314708 0.81882181 0.73577577 0.71927717 0.6127428
 0.64716751 0.72615997 0.83725712]
Layer 7
accuracy: [0.94439202 0.89699412 0.79267481 0.73360921 0.73444614 0.63848719
 0.66761119 0.76368283 0.86640236]
Layer 8
accuracy: [0.94314293 0.88979262 0

In [20]:
compute_stats(salt)

(array([[0.81915698, 0.61896351, 0.81416286],
        [0.81168458, 0.62777066, 0.73738736],
        [0.81890141, 0.6210846 , 0.71429843],
        [0.81499375, 0.63491375, 0.69612049],
        [0.870289  , 0.66757933, 0.69355002],
        [0.87140539, 0.67992581, 0.72852282],
        [0.8878925 , 0.68963187, 0.738956  ],
        [0.87827178, 0.69953657, 0.76792622],
        [0.879963  , 0.73578231, 0.78226747],
        [0.88276409, 0.73813726, 0.80328751],
        [0.9501672 , 0.83800371, 0.92135949]]),
 array([[0.08833742, 0.01342783, 0.08807199],
        [0.08063985, 0.01865393, 0.07521906],
        [0.0922654 , 0.02463489, 0.07710844],
        [0.08906258, 0.02765661, 0.08364877],
        [0.07030802, 0.03701193, 0.07521873],
        [0.06652666, 0.03482164, 0.07107541],
        [0.04971702, 0.05697026, 0.07679013],
        [0.06173024, 0.0495114 , 0.08049881],
        [0.0537024 , 0.02784152, 0.08523832],
        [0.05841574, 0.01402709, 0.08239699],
        [0.02983571, 0.01160936,

In [21]:
SAVE_FILE='e-ViT-8.pkl'

In [22]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [2]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [3]:
SAVE_FILE='e-ViT-8.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [4]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [5]:
mean_var

{'base': {'mean': array([0.65252291, 0.70290786, 0.7030379 , 0.70296154, 0.75338708,
         0.77950626, 0.80311072, 0.80952937, 0.83337525, 0.87462268,
         0.99992456]),
  'std': array([0.1256769 , 0.09817687, 0.1056688 , 0.09964658, 0.10786989,
         0.09392746, 0.09946111, 0.08701607, 0.07452171, 0.06073333,
         0.00015259]),
  'min': array([0.47676442, 0.58358716, 0.57612305, 0.58826511, 0.60268477,
         0.65954264, 0.64892579, 0.67630627, 0.6965271 , 0.77984551,
         0.99953747]),
  'max': array([0.88193417, 0.89422617, 0.91669244, 0.90566943, 0.95067415,
         0.94607315, 0.94960373, 0.94629256, 0.94728545, 0.97378386,
         1.        ])},
 'gauss': {'mean': array([0.75979367, 0.73570857, 0.72948943, 0.72226845, 0.73701153,
         0.74811791, 0.75297111, 0.76145877, 0.77329641, 0.7777059 ,
         0.84897696]),
  'std': array([0.1161841 , 0.10664938, 0.11436364, 0.11360641, 0.11151546,
         0.10581472, 0.1096565 , 0.10652691, 0.09748484, 0.09610